# <font size="5">Load and display structural examples from Jackson's data

# <font size="5">Initial Imports

In [ ]:
import numpy as np
import pickle
import os
import sys
import copy
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 2**128
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ["Arial"]
plt.rcParams['pdf.fonttype'] = 42
from matplotlib.widgets import PolygonSelector,RectangleSelector
from matplotlib.backends.backend_pdf import PdfPages
import warnings
import importlib
import jsimg as jsi
import stackview
from IPython.display import HTML, display, clear_output
from scipy.ndimage import rotate, gaussian_filter
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle
from io import BytesIO
from tqdm.auto import tqdm
import sled_general_tools
import sled_image_tools
import sled_summarize_anms
import sled_pooling
import sled_ROI_tools
import sled_reg_tools
import sled_segment_tools
import sled_movie_trace_viewer
import sled_NMF_summary
################################################################################
#sled initialization
sled_controls, tracker_info, SLURM_tracker_info, error_tracking = sled_general_tools.sled_init()
sled_controls, tracker_info, SLURM_tracker_info = sled_general_tools.where_am_I(sled_controls, tracker_info, SLURM_tracker_info)
################################################################################

In [ ]:
DS3 = "/export/general/"
DS4 = "/export/raw/"
# DS3 = "/mnt/k/"
# DS4 = "/mnt/i/"
os.path.exists(DS3)

# <font size="5">Zstack projections vs pulse splitting

In [ ]:
dend_PS_file =  os.path.join(DS3,"calcium/B00002213921/231108/run1/PoolTIF/B00002213921_GC8m_231108_ALM_PS4_Dend_Mov_1_NoShift_Reg.tif")
dend_PS_importFrs = list(range(0,175*2))
dend_PS_importFrs = list(range(0,1000))
dend_PS_data,dend_PS_importFrs = sled_image_tools.simple_frame_import(dend_PS_file,dend_PS_importFrs)

In [ ]:
soma_gauss_file =  os.path.join(DS3,"calcium/B00002213921/231109/run1/PoolTIF/B00002213921_GC8m_231109_ALM_Gauss_Soma_Mov_1_NoShift_Reg.tif")
soma_gauss_importFrs = list(range(0,175))
soma_gauss_data,soma_gauss_importFrs = sled_image_tools.simple_frame_import(soma_gauss_file,soma_gauss_importFrs)

In [ ]:
dend_zstack_files = [os.path.join(DS4,"imaging/B00002213921/zstack/2um_steps_00001_00010.tif"),\
                     os.path.join(DS4,"imaging/B00002213921/zstack/2um_steps_00001_00012.tif"),\
                     os.path.join(DS4,"imaging/B00002213921/zstack/2um_steps_00001_00014.tif"),\
                     os.path.join(DS4,"imaging/B00002213921/zstack/2um_steps_00001_00016.tif")]
dend_zstack_zpos = [10,12,14,16]
dend_zstack_files = [os.path.join(DS4,"imaging/B00002213921/zstack/2um_steps_00001_00010.tif"),\
                     os.path.join(DS4,"imaging/B00002213921/zstack/2um_steps_00001_00012.tif"),\
                     os.path.join(DS4,"imaging/B00002213921/zstack/2um_steps_00001_00014.tif")]
dend_zstack_zpos = [10,13,16]
dend_zstack_zidx = [9,12,15]

dend_zstack_files = [os.path.join(DS4,"imaging/B00002213921/zstack/2um_steps_00001_00012.tif"),\
                     os.path.join(DS4,"imaging/B00002213921/zstack/2um_steps_00001_00015.tif"),\
                     os.path.join(DS4,"imaging/B00002213921/zstack/2um_steps_00001_00018.tif")]
dend_zstack_zpos = [12,15,18]
dend_zstack_zidx = [11,14,17]

dend_zstack_importFrs = []
dend_zstack_data = {}
for g,dend_zstack_file in enumerate(dend_zstack_files):
    dend_zstack_data[g],dend_zstack_importFrs = sled_image_tools.simple_frame_import(dend_zstack_file,dend_zstack_importFrs)


In [ ]:
soma_zstack_files = [os.path.join(DS4,"imaging/B00002213921/zstack/2um_steps_00001_00230.tif")]
soma_zstack_zpos = [230]
soma_zstack_zidx = [229]
soma_zstack_importFrs = []
soma_zstack_data = {}
for g,soma_zstack_file in enumerate(soma_zstack_files):
    soma_zstack_data[g],soma_zstack_importFrs = sled_image_tools.simple_frame_import(soma_zstack_file,soma_zstack_importFrs)


In [ ]:
zstack_XZ_file = os.path.join(DS3,"calcium/B00002213921/zstack/Projections/B00002213921_GC8m_231120_ALM_Gauss_Full_ZStack_3D_Mean_Max_XZ.tif")
zstack_XZ_importFrs = []
zstack_XZ_data,soma_zstack_importFrs = sled_image_tools.simple_frame_import(zstack_XZ_file,zstack_XZ_importFrs)
zstack_XZ_data = np.squeeze(zstack_XZ_data)
zstack_YZ_file = os.path.join(DS3,"calcium/B00002213921/zstack/Projections/B00002213921_GC8m_231120_ALM_Gauss_Full_ZStack_3D_Mean_Max_YZ.tif")
zstack_YZ_importFrs = []
zstack_YZ_data,soma_zstack_importFrs = sled_image_tools.simple_frame_import(zstack_YZ_file,zstack_YZ_importFrs)
zstack_YZ_data = np.squeeze(zstack_YZ_data)

In [ ]:
scaleColor = (1,1,1)
imgColor = (1,1,1)
colorScalar = 1000

dend_PS_cont = [0,5]
soma_gauss_cont = [0,10]
dend_zstack_cont = [0,8]
dend_zstack_sum_cont = [0,20]
dend_zstack_cropX = [105,435]
print(dend_zstack_cropX[1]-dend_zstack_cropX[0])
dend_zstack_cropY = [80,410]
print(dend_zstack_cropY[1]-dend_zstack_cropY[0])
dend_zstack_cropX_xshift = 72
dend_zstack_cropY_xshift = 105

soma_zstack_cont = [0,5]
soma_zstack_cropX = [0,512]
soma_zstack_cropY = [0,512]
soma_zstack_flipLR = True
soma_zstack_cropX_xshift = 30.2
soma_zstack_cropY_xshift = 74.8

export_cmap,_,_=sled_image_tools.generate_cmap(imgColor,colorScalar)
dend_PS_img = sled_image_tools.convert2RGB(np.nanmean(dend_PS_data,axis = 0),\
        dend_PS_cont[0],dend_PS_cont[1],export_cmap,colorScalar)
soma_gauss_img = sled_image_tools.convert2RGB(np.nanmean(soma_gauss_data,axis = 0),\
        soma_gauss_cont[0],soma_gauss_cont[1],export_cmap,colorScalar)
dend_zstack_img = {}
g = 0
dend_zstack_sum = np.zeros_like(np.nanmean(dend_zstack_data[g][:,dend_zstack_cropY[0]:dend_zstack_cropY[1],dend_zstack_cropX[0]:dend_zstack_cropX[1]],axis = 0))
for g,dend_zstack_file in enumerate(dend_zstack_files):
    tempImg = np.nanmean(dend_zstack_data[g][:,dend_zstack_cropY[0]:dend_zstack_cropY[1],dend_zstack_cropX[0]:dend_zstack_cropX[1]],axis = 0)
    dend_zstack_img[g] = sled_image_tools.convert2RGB(tempImg,\
        dend_zstack_cont[0],dend_zstack_cont[1],export_cmap,colorScalar)
    dend_zstack_sum = dend_zstack_sum + tempImg
dend_zstack_sum = sled_image_tools.convert2RGB(dend_zstack_sum,\
    dend_zstack_sum_cont[0],dend_zstack_sum_cont[1],export_cmap,colorScalar)
soma_zstack_img = {}
for g,soma_zstack_file in enumerate(soma_zstack_files):
    soma_zstack_img[g] = sled_image_tools.convert2RGB(np.nanmean(soma_zstack_data[g][:,soma_zstack_cropY[0]:soma_zstack_cropY[1],soma_zstack_cropX[0]:soma_zstack_cropX[1]],axis = 0),\
        soma_zstack_cont[0],soma_zstack_cont[1],export_cmap,colorScalar)
    if soma_zstack_flipLR:
        soma_zstack_img[g] = np.fliplr(soma_zstack_img[g])

#dend pulse split at 25X 512x512 fill fraction 0.4 zoom 1x  
#somas gauss at      16X 265x256 fill fraction 0.8 zoom 1x 0.64125  
#Zstack gauss at     16X 512/512 fill fraction 0.8 zoom 1x 0.64125  

In [ ]:
zstack_umz = 2
# dend_PS_umpx = 0.4104
# soma_gauss_umpx = 0.5469
# zstack_umpx = 0.27345 #soma_gauss_umpx/2
# zstack_umpx = 0.27345 #soma_gauss_umpx/2
dend_PS_umpx = 0.4104
zstack_umpx = (dend_PS_umpx*0.0625)/0.04
soma_gauss_umpx = zstack_umpx*2
zstack_XZ_size_X_um = (zstack_XZ_data.shape[1]*zstack_umpx)
zstack_XZ_size_Y_um = (zstack_XZ_data.shape[0]*zstack_umz)
zstack_XZ_aspect = zstack_XZ_size_Y_um / zstack_XZ_size_X_um
zstack_XZ_aspect = zstack_umz / zstack_umpx
zstack_YZ_size_X_um = (zstack_YZ_data.shape[1]*zstack_umpx)
zstack_YZ_size_Y_um = (zstack_YZ_data.shape[0]*zstack_umz)
zstack_YZ_aspect = zstack_YZ_size_Y_um / zstack_YZ_size_X_um
zstack_YZ_aspect = zstack_umz / zstack_umpx


print("dend_PS_umpx = "+str(dend_PS_umpx))
print("soma_gauss_umpx = "+str(soma_gauss_umpx))
print("zstack_umpx = "+str(zstack_umpx))
print("zstack_XZ_size_X_um = "+str(zstack_XZ_size_X_um))
print("zstack_XZ_size_Y_um = "+str(zstack_XZ_size_Y_um))
print("zstack_XZ_aspect = "+str(zstack_XZ_aspect))

print("zstack_YZ_size_X_um = "+str(zstack_YZ_size_X_um))
print("zstack_YZ_size_Y_um = "+str(zstack_YZ_size_Y_um))
print("zstack_YZ_aspect = "+str(zstack_YZ_aspect))

scaleBarParams={}
scaleBarParams['orientation']='horz'
scaleBarParams['corner']='BL'
scaleBarParams['length_um'] = 20
scaleBarParams['lw'] = 1
scaleBarParams['vertAdjust']=0.03
scaleBarParams['horzAdjust']=0.04
scaleBarParams['fontsize'] = 6
scaleBarParams['includeLabel'] = False

scaleBar1Params={}
scaleBar1Params['orientation']='vert'
scaleBar1Params['corner']='BR'
scaleBar1Params['length_um'] = 100
scaleBar1Params['lw'] = 1
scaleBar1Params['vertAdjust']=0.03
scaleBar1Params['horzAdjust']=0.04
scaleBar1Params['fontsize'] = 6
scaleBar1Params['includeLabel'] = False


In [ ]:
zstack_XZ_cont = [0,40]
zstack_XZ_img = sled_image_tools.convert2RGB(zstack_XZ_data,\
        zstack_XZ_cont[0],zstack_XZ_cont[1],export_cmap,colorScalar)
plt.figure()
plt.imshow(zstack_XZ_img,interpolation='none',aspect=zstack_XZ_aspect)

In [ ]:
zstack_YZ_cont = [0,40]
zstack_YZ_img = sled_image_tools.convert2RGB(zstack_YZ_data,\
        zstack_YZ_cont[0],zstack_XZ_cont[1],export_cmap,colorScalar)
plt.figure()
plt.imshow(zstack_YZ_img,interpolation='none',aspect=zstack_YZ_aspect)

In [ ]:
importlib.reload(sled_image_tools)
figSaveDir = os.path.join("/export/general/pooledData/Calcium/JS_figsPanels/Fig2/")
figName = 'B00002213921_zstack_proj.pdf'
fontsize=6
lw = 0.5
plotScalar = 1.5
nRows= 2
nCols = 2
fig,ax = sled_general_tools.clean_subplots(nRows,nCols,figsize=(nCols*plotScalar,nRows*plotScalar))
col = 0
row = 0
ax[row,col].imshow(zstack_XZ_img,interpolation='none',aspect=zstack_XZ_aspect)
scaleBarParams['length_px'] = float(scaleBarParams['length_um'] / zstack_umpx)
# print(scaleBarParams['length_um'])
# print(zstack_umpx)
# print(scaleBarParams['length_px'])
scaleBar1Params['length_px'] = float(scaleBar1Params['length_um'] / zstack_umz)
# print(scaleBar1Params['length_um'])
# print(zstack_umz)
# print(scaleBar1Params['length_px'])
for z in dend_zstack_zidx:
    ax[row,col].plot(dend_zstack_cropX_xshift+np.array(dend_zstack_cropX),[z,z],color = (1,0,0),alpha = 0.5, lw = lw)
for z in soma_zstack_zidx:
    ax[row,col].plot(soma_zstack_cropX_xshift+np.array(soma_zstack_cropX),[z,z],color = (1,0,0),alpha = 0.5, lw = lw)
ax[row,col],scaleBarParams = sled_image_tools.add_image_scaleBar(ax[row,col], scaleBarParams, color = scaleColor, includeLabel = True)
ax[row,col],scaleBar1Params = sled_image_tools.add_image_scaleBar(ax[row,col], scaleBar1Params, color = scaleColor, includeLabel = True)
ax[row,col].text(5,0,"XZ Max Proj.",ha='left',va='bottom',color=(0,0,0),fontsize=fontsize)
ax[row,col].text(zstack_XZ_img.shape[1]-5,5,"("+str(zstack_XZ_cont[0])+"-"+str(zstack_XZ_cont[1])+")",ha='right',va='top',color=scaleColor,fontsize=fontsize)
ax[row,col]=sled_image_tools.imshow_cleanup(ax[row,col])

col = 1
ax[row,col].imshow(zstack_YZ_img,interpolation='none',aspect=zstack_YZ_aspect)
for z in dend_zstack_zidx:
    ax[row,col].plot(dend_zstack_cropY_xshift+np.array(dend_zstack_cropY),[z,z],color = (1,0,0),alpha = 0.5, lw = lw)
for z in soma_zstack_zidx:
    ax[row,col].plot(soma_zstack_cropY_xshift+np.array(soma_zstack_cropY),[z,z],color = (1,0,0),alpha = 0.5, lw = lw)
ax[row,col],scaleBarParams = sled_image_tools.add_image_scaleBar(ax[row,col], scaleBarParams, color = scaleColor, includeLabel = True)
ax[row,col],scaleBar1Params = sled_image_tools.add_image_scaleBar(ax[row,col], scaleBar1Params, color = scaleColor, includeLabel = True)
ax[row,col].text(5,0,"YZ Max Proj.",ha='left',va='bottom',color=(0,0,0),fontsize=fontsize)
ax[row,col].text(zstack_YZ_img.shape[1]-5,5,"("+str(zstack_YZ_cont[0])+"-"+str(zstack_YZ_cont[1])+")",ha='right',va='top',color=scaleColor,fontsize=fontsize)
ax[row,col]=sled_image_tools.imshow_cleanup(ax[row,col])

row = 1
for col in range(nCols):
    fig.delaxes(ax[row,col])

plt.show()
with PdfPages(os.path.join(figSaveDir,figName)) as pdf:
    pdf.savefig(fig,bbox_inches='tight',pad_inches=0.05,dpi=600)  # saves the current figure into a pdf page

In [ ]:
figSaveDir = os.path.join("/export/general/pooledData/Calcium/JS_figsPanels/Fig2/")
figName = 'B00002213921_pulse_split_vs_zstack_3slice.pdf'
plotScalar = 0.7
nRows= 4
nCols = 2
fig,ax = sled_general_tools.clean_subplots(nRows,nCols,figsize=(nCols*plotScalar,nRows*plotScalar))
row = -1
col = 0
for g,dend_zstack_file in enumerate(dend_zstack_files):
    row+=1
    ax[row,col].imshow(dend_zstack_img[g],interpolation='none')
    scaleBarParams['length_px'] = float(scaleBarParams['length_um'] / zstack_umpx)
    ax[row,col],scaleBarParams = sled_image_tools.add_image_scaleBar(ax[row,col], scaleBarParams, color = scaleColor, includeLabel = True)
    ax[row,col].text(5,5,"Z-stack\nz = +"+str(zstack_umz*dend_zstack_zpos[g])+" μm",ha='left',va='top',color=scaleColor,fontsize=fontsize)
    ax[row,col].text(dend_zstack_img[g].shape[1]-5,5,"("+str(dend_zstack_cont[0])+"-"+str(dend_zstack_cont[1])+")",ha='right',va='top',color=scaleColor,fontsize=fontsize)
    ax[row,col]=sled_image_tools.imshow_cleanup(ax[row,col])
while g+1 < nRows:
    g+=1
    row = g
    fig.delaxes(ax[row,col])
row = 0
col = 1
ax[row,col].imshow(dend_PS_img,interpolation='none')
scaleBarParams['length_px'] = float(scaleBarParams['length_um'] / dend_PS_umpx)
ax[row,col],scaleBarParams = sled_image_tools.add_image_scaleBar(ax[row,col], scaleBarParams, color = scaleColor, includeLabel = True)
ax[row,col].text(5,5,"Pulse Split\nX4",ha='left',va='top',color=scaleColor,fontsize=fontsize)
ax[row,col].text(dend_PS_img.shape[1]-5,5,"("+str(dend_PS_cont[0])+"-"+str(dend_PS_cont[1])+")",ha='right',va='top',color=scaleColor,fontsize=fontsize)
ax[row,col]=sled_image_tools.imshow_cleanup(ax[row,col])
plt.subplots_adjust(wspace=0.01, hspace=0.01)         
row = 1
ax[row,col].imshow(dend_zstack_sum,interpolation='none')
scaleBarParams['length_px'] = float(scaleBarParams['length_um'] / zstack_umpx)
ax[row,col],scaleBarParams = sled_image_tools.add_image_scaleBar(ax[row,col], scaleBarParams, color = scaleColor, includeLabel = True)
ax[row,col].text(5,5,"Z-stack\nSum +"+str(zstack_umz*dend_zstack_zpos[0])+"\n+"+str(zstack_umz*dend_zstack_zpos[-1])+" μm",ha='left',va='top',color=scaleColor,fontsize=fontsize)
ax[row,col].text(dend_zstack_sum.shape[1]-5,5,"("+str(dend_zstack_sum_cont[0])+"-"+str(dend_zstack_sum_cont[1])+")",ha='right',va='top',color=scaleColor,fontsize=fontsize)
ax[row,col]=sled_image_tools.imshow_cleanup(ax[row,col])
plt.subplots_adjust(wspace=0.01, hspace=0.01) 

row = 2
ax[row,col].imshow(soma_gauss_img,interpolation='none')
scaleBarParams['length_px'] = float(scaleBarParams['length_um'] / soma_gauss_umpx)
ax[row,col],scaleBarParams = sled_image_tools.add_image_scaleBar(ax[row,col], scaleBarParams, color = scaleColor, includeLabel = True)
ax[row,col].text(5,5,"Soma\nGaussian",ha='left',va='top',color=scaleColor,fontsize=fontsize)
ax[row,col].text(soma_gauss_img.shape[1]-5,5,"("+str(soma_gauss_cont[0])+"-"+str(soma_gauss_cont[1])+")",ha='right',va='top',color=scaleColor,fontsize=fontsize)
ax[row,col]=sled_image_tools.imshow_cleanup(ax[row,col])
plt.subplots_adjust(wspace=0.01, hspace=0.01)         
row = 3
g = 0
ax[row,col].imshow(soma_zstack_img[g],interpolation='none')
scaleBarParams['length_px'] = float(scaleBarParams['length_um'] / zstack_umpx)
ax[row,col],scaleBarParams = sled_image_tools.add_image_scaleBar(ax[row,col], scaleBarParams, color = scaleColor, includeLabel = True)
ax[row,col].text(5,5,"Z-stack\nz = +"+str(zstack_umz*soma_zstack_zpos[g])+" μm",ha='left',va='top',color=scaleColor,fontsize=fontsize)
ax[row,col].text(soma_zstack_img[g].shape[1]-5,5,"("+str(soma_zstack_cont[0])+"-"+str(soma_zstack_cont[1])+")",ha='right',va='top',color=scaleColor,fontsize=fontsize)
ax[row,col]=sled_image_tools.imshow_cleanup(ax[row,col])
plt.subplots_adjust(wspace=0.02, hspace=0.02) 

plt.show()
with PdfPages(os.path.join(figSaveDir,figName)) as pdf:
    pdf.savefig(fig,bbox_inches='tight',pad_inches=0.05,dpi=600)  # saves the current figure into a pdf page